# (노트) 판다스2022
- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- categories: [데이터과학]

## import 

In [6]:
import numpy as np
import pandas as pd

## 부분 데이터 꺼내기: 판다스를 왜 써야할까?

### 기본 인덱싱

`-` 예제1: 기본인덱싱

In [7]:
a='asdf' 
a[2]

'd'

In [8]:
a='asdf' 
a[-1]

'f'

`-` 예제2: 슬라이싱

In [9]:
a='asdf'
a[1:3]

'sd'

In [10]:
a='asdf'
a[-2:]

'df'

`-` 예제3: 스트라이딩

In [11]:
a='asdfg'
a[::2]

'adg'

`-` 예제4: 불가능한것

In [12]:
a='asdf' 
a[[1,2]] # 정수인덱스의 리스트를 전달하여 인덱싱해보자 -> 실패! 

TypeError: string indices must be integers

In [13]:
a='asdf'
a[[True,False,True,False]] # bool로 이루어진 리스트를 전달하여 인덱싱해보자 -> 실패!

TypeError: string indices must be integers

### 팬시인덱싱

`-` 예제1: 인덱스의 리스트 (혹은 ndarray)를 전달

In [14]:
a=np.arange(5)
a[[1,2,-1]]

array([1, 2, 4])

In [15]:
a[np.array([1,2,-1])]

array([1, 2, 4])

`-` 예제2: bool로 이루어진 리스트 (혹은 ndarray)를 전달 

In [16]:
a=np.arange(5)
a[[True,True,False,False,False]]

array([0, 1])

In [17]:
a[np.array([True,True,False,False,False])]

array([0, 1])

`-` 예제3: bool로 이루어진 리스트 (혹은 ndarray)를 전달 

In [18]:
a=np.arange(5)
a[a<3]

array([0, 1, 2])

### 2차원자료형의 인덱싱 

`-` 예제1

In [19]:
a= np.arange(4*3).reshape(4,3)
a

array([[ 0,  1,  2],
       [ 3,  4,  5],
       [ 6,  7,  8],
       [ 9, 10, 11]])

In [20]:
a[0:2,1]

array([1, 4])

`-` 예제2: 차원을 유지하면서 인덱싱

In [21]:
a[0:2,[1]]

array([[1],
       [4]])

### Hash

`-` 예제1: (key, value)

In [22]:
d={'att':65, 'rep':45, 'mid':30, 'fin':100} 
d

{'att': 65, 'rep': 45, 'mid': 30, 'fin': 100}

In [23]:
d['att'] # key를 넣으면 value가 리턴

65

`-` 예제2: numpy와 비교

In [24]:
np.random.seed(43052)
att = np.random.choice(np.arange(10,21)*5,200)
rep = np.random.choice(np.arange(5,21)*5,200)
mid = np.random.choice(np.arange(0,21)*5,200)
fin = np.random.choice(np.arange(0,21)*5,200)
key = ['202212'+str(s) for s in np.random.choice(np.arange(300,501),200,replace=False)]
test_dic = {key[i] : {'att':att[i], 'rep':rep[i], 'mid':mid[i], 'fin':fin[i]} for i in range(200)}

학번 '202212354' 에 해당하는 학생의 출석점수를 알고싶다면? 

(풀이1)

In [25]:
test_dic['202212354']['att']

50

(풀이2)

In [26]:
test_ndarray = np.array([key,att,rep,mid,fin],dtype=np.int64).T

In [27]:
test_ndarray[test_ndarray[:,0] == 202212354,1] # 이게 무슨코드야 도데체! 

array([50])

**(풀이2)가 (풀이1)과 비교하여 불편한점**
- test_ndarray에서 첫번째 컬럼은 student id, 두번째 칼럼은 att라는 사실을 암기하고 있어야 한다. 
- student id가 아니고 만약 학생이름을 쓴다면 모든 자료형은 문자형이 되어야 한다. 
- 작성한 코드의 가독성이 없다. (위치로 접근하기 때문) 

`-` 요약: hash 스타일로 정보를 추출하는것이 유용할 때가 있다. 그리고 보통 hash스타일로 정보를 뽑는것이 유리하다. (사실 numpy는 정보추출을 위해 개발된 자료형이 아니라 행렬 및 벡터의 수학연산을 지원하기 위한 자료형이다) 

`-` 소망: 정보를 추출할때는 hash 스타일이 유리한것은 알겠음 $\to$ 하지만 나는 가끔씩 numpy스타일로 원하는 정보를 뽑고 싶은걸? 그리고 딕셔너리 형태가 아니고 엑셀처럼(행렬처럼) 데이터를 보고 싶은걸? $\to$ pandas의 개발 

## pandas의 개발의 동기 

### 일단 엑셀처럼(행렬처럼) 데이터를 보고 싶다

In [28]:
np.random.seed(43052)
att = np.random.choice(np.arange(10,21)*5,20)
rep = np.random.choice(np.arange(5,21)*5,20)
mid = np.random.choice(np.arange(0,21)*5,20)
fin = np.random.choice(np.arange(0,21)*5,20)
key = ['202212'+str(s) for s in np.random.choice(np.arange(300,501),20,replace=False)]
test_dic = {key[i] : {'att':att[i], 'rep':rep[i], 'mid':mid[i], 'fin':fin[i]} for i in range(20)}

In [29]:
test_dic # 그다지 가독성이 없어.. 테이블 형태로 보고싶다!

{'202212380': {'att': 65, 'rep': 55, 'mid': 50, 'fin': 40},
 '202212370': {'att': 95, 'rep': 100, 'mid': 50, 'fin': 80},
 '202212363': {'att': 65, 'rep': 90, 'mid': 60, 'fin': 30},
 '202212488': {'att': 55, 'rep': 80, 'mid': 75, 'fin': 80},
 '202212312': {'att': 80, 'rep': 30, 'mid': 30, 'fin': 100},
 '202212377': {'att': 75, 'rep': 40, 'mid': 100, 'fin': 15},
 '202212463': {'att': 65, 'rep': 45, 'mid': 45, 'fin': 90},
 '202212471': {'att': 60, 'rep': 60, 'mid': 25, 'fin': 0},
 '202212400': {'att': 95, 'rep': 65, 'mid': 20, 'fin': 10},
 '202212469': {'att': 90, 'rep': 80, 'mid': 80, 'fin': 20},
 '202212318': {'att': 55, 'rep': 75, 'mid': 35, 'fin': 25},
 '202212432': {'att': 95, 'rep': 95, 'mid': 45, 'fin': 0},
 '202212443': {'att': 95, 'rep': 55, 'mid': 15, 'fin': 35},
 '202212367': {'att': 50, 'rep': 80, 'mid': 40, 'fin': 30},
 '202212458': {'att': 50, 'rep': 55, 'mid': 15, 'fin': 85},
 '202212396': {'att': 95, 'rep': 30, 'mid': 30, 'fin': 95},
 '202212482': {'att': 50, 'rep': 50, 'm

(방법1)

In [30]:
test_ndarray = np.array([key,att,rep,mid,fin],dtype=np.int64).T
test_ndarray

array([[202212380,        65,        55,        50,        40],
       [202212370,        95,       100,        50,        80],
       [202212363,        65,        90,        60,        30],
       [202212488,        55,        80,        75,        80],
       [202212312,        80,        30,        30,       100],
       [202212377,        75,        40,       100,        15],
       [202212463,        65,        45,        45,        90],
       [202212471,        60,        60,        25,         0],
       [202212400,        95,        65,        20,        10],
       [202212469,        90,        80,        80,        20],
       [202212318,        55,        75,        35,        25],
       [202212432,        95,        95,        45,         0],
       [202212443,        95,        55,        15,        35],
       [202212367,        50,        80,        40,        30],
       [202212458,        50,        55,        15,        85],
       [202212396,        95,        30,

(방법2)

In [32]:
pd.DataFrame(test_dic).T

,att,rep,mid,fin
202212380,65,55,50,40
202212370,95,100,50,80
202212363,65,90,60,30
202212488,55,80,75,80
202212312,80,30,30,100
202212377,75,40,100,15
202212463,65,45,45,90
202212471,60,60,25,0
202212400,95,65,20,10
202212469,90,80,80,20


(방법3)

In [35]:
test_dic2 = {'att':{key[i]:att[i] for i in range(20)},
             'rep':{key[i]:rep[i] for i in range(20)},
             'mid':{key[i]:mid[i] for i in range(20)},
             'fin':{key[i]:fin[i] for i in range(20)}}

In [36]:
pd.DataFrame(test_dic2)

,att,rep,mid,fin
202212380,65,55,50,40
202212370,95,100,50,80
202212363,65,90,60,30
202212488,55,80,75,80
202212312,80,30,30,100
202212377,75,40,100,15
202212463,65,45,45,90
202212471,60,60,25,0
202212400,95,65,20,10
202212469,90,80,80,20


(방법4)

In [37]:
df= pd.DataFrame({'att':att,'rep':rep,'mid':mid,'fin':fin},index=key)
df

,att,rep,mid,fin
202212380,65,55,50,40
202212370,95,100,50,80
202212363,65,90,60,30
202212488,55,80,75,80
202212312,80,30,30,100
202212377,75,40,100,15
202212463,65,45,45,90
202212471,60,60,25,0
202212400,95,65,20,10
202212469,90,80,80,20


(방법5)

In [46]:
df= pd.DataFrame({'att':att,'rep':rep,'mid':mid,'fin':fin})
df

,att,rep,mid,fin
0,65,55,50,40
1,95,100,50,80
2,65,90,60,30
3,55,80,75,80
4,80,30,30,100
5,75,40,100,15
6,65,45,45,90
7,60,60,25,0
8,95,65,20,10
9,90,80,80,20


In [48]:
df=df.set_index([key])
df

,att,rep,mid,fin
202212380,65,55,50,40
202212370,95,100,50,80
202212363,65,90,60,30
202212488,55,80,75,80
202212312,80,30,30,100
202212377,75,40,100,15
202212463,65,45,45,90
202212471,60,60,25,0
202212400,95,65,20,10
202212469,90,80,80,20


### 해싱으로 원하는 정보를 뽑으면 좋겠다 (마치 딕셔너리처럼)

`-` 예제1: 출석점수를 출력 

In [65]:
test_dic2['att']

{'202212380': 65,
 '202212370': 95,
 '202212363': 65,
 '202212488': 55,
 '202212312': 80,
 '202212377': 75,
 '202212463': 65,
 '202212471': 60,
 '202212400': 95,
 '202212469': 90,
 '202212318': 55,
 '202212432': 95,
 '202212443': 95,
 '202212367': 50,
 '202212458': 50,
 '202212396': 95,
 '202212482': 50,
 '202212452': 65,
 '202212387': 70,
 '202212354': 90}

In [66]:
df['att']

202212380    65
202212370    95
202212363    65
202212488    55
202212312    80
202212377    75
202212463    65
202212471    60
202212400    95
202212469    90
202212318    55
202212432    95
202212443    95
202212367    50
202212458    50
202212396    95
202212482    50
202212452    65
202212387    70
202212354    90
Name: att, dtype: int64

`-` 예제2: 학번 202212380 의 출석점수 출력 

In [61]:
df['att']['202212380'] # 거의 반 딕셔너리

65

In [63]:
test_dic2['att']['202212380']

65

### 때로는 인덱싱으로 원하는 정보를 뽑고 싶다 (마치 리스트나 넘파이처럼)

`-` 예제1: 첫번째 학생의 기말고사 성적을 출력하고 싶다. 

In [128]:
test_ndarray[0,-1]

40

In [129]:
df.iloc[0,-1]

40

- 벼락치기: df에서 iloc 특수기능을 이용하면 넘파이 인덱싱처럼 원소출력이 가능하다 

`-` 예제2: 홀수번째 학생의 점수 출력 

In [130]:
test_ndarray[::2]

array([[202212380,        65,        55,        50,        40],
       [202212363,        65,        90,        60,        30],
       [202212312,        80,        30,        30,       100],
       [202212463,        65,        45,        45,        90],
       [202212400,        95,        65,        20,        10],
       [202212318,        55,        75,        35,        25],
       [202212443,        95,        55,        15,        35],
       [202212458,        50,        55,        15,        85],
       [202212482,        50,        50,        45,        10],
       [202212387,        70,        70,        40,        35]])

In [131]:
df.iloc[::2]

,att,rep,mid,fin
202212380,65,55,50,40
202212363,65,90,60,30
202212312,80,30,30,100
202212463,65,45,45,90
202212400,95,65,20,10
202212318,55,75,35,25
202212443,95,55,15,35
202212458,50,55,15,85
202212482,50,50,45,10
202212387,70,70,40,35


`-` 예제3: 맽 끝에서 3명의 점수 출력

In [132]:
test_ndarray[-3:]

array([[202212452,        65,        55,        15,        45],
       [202212387,        70,        70,        40,        35],
       [202212354,        90,        90,        80,        90]])

In [133]:
df.iloc[-3:]

,att,rep,mid,fin
202212452,65,55,15,45
202212387,70,70,40,35
202212354,90,90,80,90


`-` 예제4: 맨 끝에서 3번명의 마지막 2칼럼을 출력 

In [134]:
test_ndarray[-3:,-2:]

array([[15, 45],
       [40, 35],
       [80, 90]])

In [135]:
df.iloc[-3:,-2:]

,mid,fin
202212452,15,45
202212387,40,35
202212354,80,90


### 궁극: 해싱과 인덱싱 모두 지원하는 아주 우수한 자료형을 만들고 싶음

`-` 예제1: 중간고사 점수가 20점 이상이면서 동시에 출석점수가 60점미만인 학생들의 기말고사 점수를 출력 

(방법1) 데이터베이스 스타일

In [99]:
df.query("mid >= 20 and att <60")

,att,rep,mid,fin
202212488,55,80,75,80
202212318,55,75,35,25
202212367,50,80,40,30
202212482,50,50,45,10


In [161]:
df.query("mid >= 20 and att < 60")['fin']

202212488    80
202212318    25
202212367    30
202212482    10
Name: fin, dtype: int64

In [162]:
df.query("mid >= 20 and att < 60")[['fin']] #데이터 프레임이 유지되면서 출력

,fin
202212488,80
202212318,25
202212367,30
202212482,10


(참고) 넘파이 스타일이라면?

In [163]:
test_ndarray

array([[202212380,        65,        55,        50,        40],
       [202212370,        95,       100,        50,        80],
       [202212363,        65,        90,        60,        30],
       [202212488,        55,        80,        75,        80],
       [202212312,        80,        30,        30,       100],
       [202212377,        75,        40,       100,        15],
       [202212463,        65,        45,        45,        90],
       [202212471,        60,        60,        25,         0],
       [202212400,        95,        65,        20,        10],
       [202212469,        90,        80,        80,        20],
       [202212318,        55,        75,        35,        25],
       [202212432,        95,        95,        45,         0],
       [202212443,        95,        55,        15,        35],
       [202212367,        50,        80,        40,        30],
       [202212458,        50,        55,        15,        85],
       [202212396,        95,        30,

In [164]:
test_ndarray[:,3]>=20 # 중간고사가 20점이상 

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True, False,  True, False,  True,  True, False,
        True,  True])

In [165]:
test_ndarray[:,1] < 60 # 출석이 60점이하 

array([False, False, False,  True, False, False, False, False, False,
       False,  True, False, False,  True,  True, False,  True, False,
       False, False])

In [166]:
(test_ndarray[:,3]>=20) & (test_ndarray[:,1] < 60)

array([False, False, False,  True, False, False, False, False, False,
       False,  True, False, False,  True, False, False,  True, False,
       False, False])

In [167]:
test_ndarray[(test_ndarray[:,3]>=20) & (test_ndarray[:,1] < 60),-1]

array([80, 25, 30, 10])

- 구현난이도 어려움, 가독성 꽝

`-` 예제2: '중간고사점수<기말고사점수' 인 학생들의 출석점수 평균을 구하자. 

In [173]:
df.query('mid<fin')

,att,rep,mid,fin
202212370,95,100,50,80
202212488,55,80,75,80
202212312,80,30,30,100
202212463,65,45,45,90
202212443,95,55,15,35
202212458,50,55,15,85
202212396,95,30,30,95
202212452,65,55,15,45
202212354,90,90,80,90


In [174]:
df.query('mid<fin')['att']

202212370    95
202212488    55
202212312    80
202212463    65
202212443    95
202212458    50
202212396    95
202212452    65
202212354    90
Name: att, dtype: int64

In [175]:
df.query('mid<fin')['att'].mean()

76.66666666666667

## pandas 사용법 (안할거에요)

### pandas 공부 1단계

#### 데이터프레임 선언 

`-` dictionary에서 만든다

In [178]:
pd.DataFrame({'att':[30,40,50],'mid':[50,60,70]})

,att,mid
0,30,50
1,40,60
2,50,70


In [179]:
pd.DataFrame({'att':(30,40,50),'mid':(50,60,70)})

,att,mid
0,30,50
1,40,60
2,50,70


In [185]:
pd.DataFrame({'att':np.array([30,40,50]),'mid':np.array([50,60,70])})

,att,mid
0,30,50
1,40,60
2,50,70


In [184]:
pd.DataFrame({'att':pd.Series([30,40,50]),'mid':pd.Series([50,60,70])})

,att,mid
0,30,50
1,40,60
2,50,70


`-` 2차원 ndarray에서 만든다 

In [188]:
np.arange(2*3).reshape(2,3)

array([[0, 1, 2],
       [3, 4, 5]])

In [189]:
pd.DataFrame(np.arange(2*3).reshape(2,3))

,0,1,2
0,0,1,2
1,3,4,5


#### 열의 이름 부여 

`-` 방법1: 딕셔너리에서 만들면 key가 자동으로 열의 이름이 된다. 

In [191]:
pd.DataFrame({'att':pd.Series([30,40,50]),'mid':pd.Series([50,60,70])})

,att,mid
0,30,50
1,40,60
2,50,70


`-` 방법2: columns=[] 옵션사용 

In [196]:
pd.DataFrame(np.arange(2*3).reshape(2,3),columns=['X1','X2','X3'])

,X1,X2,X3
0,0,1,2
1,3,4,5


`-` 방법3: df.columns 에 열이름을 덮어씀 (1)

In [231]:
df=pd.DataFrame(np.arange(2*3).reshape(2,3))

In [232]:
df

,0,1,2
0,0,1,2
1,3,4,5


In [233]:
df.columns

RangeIndex(start=0, stop=3, step=1)

In [235]:
df.columns=['X1','X2','X3'] # 추천하는 방법 아님

In [236]:
df

,X1,X2,X3
0,0,1,2
1,3,4,5


`-` 방법4: df.columns 에 열이름을 덮어씀 (2)

In [237]:
df=pd.DataFrame(np.arange(2*3).reshape(2,3))
df.columns= pd.Index(['X1','X2','X3'])
df

,X1,X2,X3
0,0,1,2
1,3,4,5


이것이 방법3보다 더 좋다. 왜? 

In [243]:
df.columns, type(df.columns)

(Index(['X1', 'X2', 'X3'], dtype='object'), pandas.core.indexes.base.Index)

In [244]:
['X1','X2','X3'], type(['X1','X2','X3'])

(['X1', 'X2', 'X3'], list)

In [245]:
pd.Index(['X1','X2','X3']), type(pd.Index(['X1','X2','X3']))

(Index(['X1', 'X2', 'X3'], dtype='object'), pandas.core.indexes.base.Index)

즉 pd.Index(['X1','X2','X3']) 이 좀 더 판다스가 처리하기 용이한 자료형태임

#### 행의 이름 부여 

`-` 방법1: 중첩 dict이면 nested dic의 key가 알아서 행의 이름으로 된다. 

In [246]:
pd.DataFrame({'att':{'guebin':30, 'iu':40, 'hynn':50},'mid':{'guebin':5, 'iu':45, 'hynn':90}})

,att,mid
guebin,30,5
iu,40,45
hynn,50,90


`-` 방법2: index 옵션 이용

In [252]:
pd.DataFrame({'att':[30,40,50],'mid':[5,45,90]},index=['guebin','iu','hynn'])

,att,mid
guebin,30,5
iu,40,45
hynn,50,90


`-` 방법3: df.index에 덮어씀

In [255]:
df=pd.DataFrame({'att':[30,40,50],'mid':[5,45,90]})
df 

,att,mid
0,30,5
1,40,45
2,50,90


In [256]:
df.index

RangeIndex(start=0, stop=3, step=1)

In [267]:
df.index = pd.Index(['guebin','iu','hynn'])
#df.index = ['guebin','iu','hynn'] <-- 이것도 가능함

In [264]:
df

,att,mid
guebin,30,5
iu,40,45
hynn,50,90


`-` 방법4: set_index 메소드 이용 

In [271]:
df=pd.DataFrame({'att':[30,40,50],'mid':[5,45,90]})
df 

,att,mid
0,30,5
1,40,45
2,50,90


In [276]:
df.set_index(['guebin','iu','hynn']) # 에러난다

KeyError: "None of ['guebin', 'iu', 'hynn'] are in the columns"

In [277]:
df.set_index([['guebin','iu','hynn']]) # 꺽쇠두번..

,att,mid
guebin,30,5
iu,40,45
hynn,50,90


In [279]:
df.set_index(pd.Index(['guebin','iu','hynn'])) # 이게 안전하고 좋다

,att,mid
guebin,30,5
iu,40,45
hynn,50,90


#### 자료형, len, shape

In [314]:
df=pd.DataFrame({'att':[30,40,50],'mid':[5,45,90]})
df 

,att,mid
0,30,5
1,40,45
2,50,90


`-` type

In [322]:
type(df)

pandas.core.frame.DataFrame

`-` len

In [315]:
len(df)

3

`-` shape

In [316]:
df.shape

(3, 2)

`-` for문의 반복변수

In [318]:
for k in df:
    print(k) # 진짜 딕셔너리같죠

att
mid


In [320]:
for k in {'att':[30,40,50],'mid':[5,45,90]}: 
    print(k)

att
mid


#### pd.Series 

`-` 2차원 ndarray가 데이터프레임에 대응한다면 1차원 ndarray는 시리즈에 대응한다. 

In [325]:
a=pd.Series(np.random.randn(10))
a

0    0.723759
1    0.217990
2    0.194022
3   -0.688990
4   -0.351670
5    0.990933
6    1.212147
7   -0.608965
8    0.032549
9   -1.884089
dtype: float64

In [328]:
type(a)

pandas.core.series.Series

In [329]:
len(a)

10

In [335]:
a.shape

(10,)

In [333]:
for value in a:
    print(value)

0.7237590624253404
0.21798967912700873
0.1940223087322443
-0.6889899757985083
-0.3516696436204985
0.9909329773184973
1.2121468150185186
-0.6089654373693767
0.03254898346416765
-1.8840890712454552


### pandas 공부 2단계 

`-` 데이터

In [336]:
df=pd.DataFrame(test_dic2)
df

,att,rep,mid,fin
202212380,65,55,50,40
202212370,95,100,50,80
202212363,65,90,60,30
202212488,55,80,75,80
202212312,80,30,30,100
202212377,75,40,100,15
202212463,65,45,45,90
202212471,60,60,25,0
202212400,95,65,20,10
202212469,90,80,80,20


#### 하나의 칼럼을 선택

`-` 방법1

In [284]:
df.att

202212380    65
202212370    95
202212363    65
202212488    55
202212312    80
202212377    75
202212463    65
202212471    60
202212400    95
202212469    90
202212318    55
202212432    95
202212443    95
202212367    50
202212458    50
202212396    95
202212482    50
202212452    65
202212387    70
202212354    90
Name: att, dtype: int64

`-` 방법2

In [286]:
df['att']

202212380    65
202212370    95
202212363    65
202212488    55
202212312    80
202212377    75
202212463    65
202212471    60
202212400    95
202212469    90
202212318    55
202212432    95
202212443    95
202212367    50
202212458    50
202212396    95
202212482    50
202212452    65
202212387    70
202212354    90
Name: att, dtype: int64

`-` 방법3

In [288]:
df[['att']]

,att
202212380,65
202212370,95
202212363,65
202212488,55
202212312,80
202212377,75
202212463,65
202212471,60
202212400,95
202212469,90


- df['att']는 series를 리턴하고 df[['att']]는 dataframe을 리턴한다.

`-` 방법4

In [290]:
df.loc[:,'att']

202212380    65
202212370    95
202212363    65
202212488    55
202212312    80
202212377    75
202212463    65
202212471    60
202212400    95
202212469    90
202212318    55
202212432    95
202212443    95
202212367    50
202212458    50
202212396    95
202212482    50
202212452    65
202212387    70
202212354    90
Name: att, dtype: int64

`-` 방법5

In [293]:
df.loc[:,['att']]

,att
202212380,65
202212370,95
202212363,65
202212488,55
202212312,80
202212377,75
202212463,65
202212471,60
202212400,95
202212469,90


- df.loc[:,'att']는 series를 리턴하고 df[:,['att']]는 dataframe을 리턴한다.

`-` 방법6

In [297]:
df.loc[:,[True,False,False,False]]

,att
202212380,65
202212370,95
202212363,65
202212488,55
202212312,80
202212377,75
202212463,65
202212471,60
202212400,95
202212469,90


`-` 방법7 

In [300]:
df.iloc[:,0]

202212380    65
202212370    95
202212363    65
202212488    55
202212312    80
202212377    75
202212463    65
202212471    60
202212400    95
202212469    90
202212318    55
202212432    95
202212443    95
202212367    50
202212458    50
202212396    95
202212482    50
202212452    65
202212387    70
202212354    90
Name: att, dtype: int64

`-` 방법8

In [299]:
df.iloc[:,[0]]

,att
202212380,65
202212370,95
202212363,65
202212488,55
202212312,80
202212377,75
202212463,65
202212471,60
202212400,95
202212469,90


`-` 방법9

In [306]:
df.iloc[:,[True,False,False,False]]

,att
202212380,65
202212370,95
202212363,65
202212488,55
202212312,80
202212377,75
202212463,65
202212471,60
202212400,95
202212469,90


#### 여러개의 칼럼을 선택 

`-` 방법1

In [339]:
df[['att','fin']]

,att,fin
202212380,65,40
202212370,95,80
202212363,65,30
202212488,55,80
202212312,80,100
202212377,75,15
202212463,65,90
202212471,60,0
202212400,95,10
202212469,90,20


`-` 방법2

In [340]:
df.loc[:,['att','fin']]

,att,fin
202212380,65,40
202212370,95,80
202212363,65,30
202212488,55,80
202212312,80,100
202212377,75,15
202212463,65,90
202212471,60,0
202212400,95,10
202212469,90,20


`-` 방법3

In [342]:
df.loc[:,'att':'mid']  ### 키인데.. 슬라이싱?

,att,rep,mid
202212380,65,55,50
202212370,95,100,50
202212363,65,90,60
202212488,55,80,75
202212312,80,30,30
202212377,75,40,100
202212463,65,45,45
202212471,60,60,25
202212400,95,65,20
202212469,90,80,80


In [352]:
df.loc[:,:'mid']  ### 첫인덱스의 생략 

,att,rep,mid
202212380,65,55,50
202212370,95,100,50
202212363,65,90,60
202212488,55,80,75
202212312,80,30,30
202212377,75,40,100
202212463,65,45,45
202212471,60,60,25
202212400,95,65,20
202212469,90,80,80


- 주의: 이때는 마지막 인덱스가 포함된다. 

In [354]:
df.loc[:,'mid':]  ### 마지막 인덱스의 생략

,mid,fin
202212380,50,40
202212370,50,80
202212363,60,30
202212488,75,80
202212312,30,100
202212377,100,15
202212463,45,90
202212471,25,0
202212400,20,10
202212469,80,20


`-` 방법4

In [343]:
df.loc[:,[True,True,True,False]]

,att,rep,mid
202212380,65,55,50
202212370,95,100,50
202212363,65,90,60
202212488,55,80,75
202212312,80,30,30
202212377,75,40,100
202212463,65,45,45
202212471,60,60,25
202212400,95,65,20
202212469,90,80,80


`-` 방법5

In [344]:
df.iloc[:,[0,1,2]]

,att,rep,mid
202212380,65,55,50
202212370,95,100,50
202212363,65,90,60
202212488,55,80,75
202212312,80,30,30
202212377,75,40,100
202212463,65,45,45
202212471,60,60,25
202212400,95,65,20
202212469,90,80,80


In [350]:
df.iloc[:,:3]

,att,rep,mid
202212380,65,55,50
202212370,95,100,50
202212363,65,90,60
202212488,55,80,75
202212312,80,30,30
202212377,75,40,100
202212463,65,45,45
202212471,60,60,25
202212400,95,65,20
202212469,90,80,80


In [349]:
df.iloc[:,range(3)]

,att,rep,mid
202212380,65,55,50
202212370,95,100,50
202212363,65,90,60
202212488,55,80,75
202212312,80,30,30
202212377,75,40,100
202212463,65,45,45
202212471,60,60,25
202212400,95,65,20
202212469,90,80,80


In [346]:
df.iloc[:,:2] # 스트라이딩..

,att,rep
202212380,65,55
202212370,95,100
202212363,65,90
202212488,55,80
202212312,80,30
202212377,75,40
202212463,65,45
202212471,60,60
202212400,95,65
202212469,90,80


`-` 방법6

In [355]:
df.iloc[:,[True,True,True,False]]

,att,rep,mid
202212380,65,55,50
202212370,95,100,50
202212363,65,90,60
202212488,55,80,75
202212312,80,30,30
202212377,75,40,100
202212463,65,45,45
202212471,60,60,25
202212400,95,65,20
202212469,90,80,80


(주의) loc에서의 슬라이싱은 마지막변수를 포함하지마 iloc에서는 포함하지 않음 

In [357]:
df.iloc[:,0:3] #0,1,2,3중 3은 포함되지 않음

,att,rep,mid
202212380,65,55,50
202212370,95,100,50
202212363,65,90,60
202212488,55,80,75
202212312,80,30,30
202212377,75,40,100
202212463,65,45,45
202212471,60,60,25
202212400,95,65,20
202212469,90,80,80


In [362]:
df.loc[:,'att':'mid'] # 'mid'가 포함된다.

,att,rep,mid
202212380,65,55,50
202212370,95,100,50
202212363,65,90,60
202212488,55,80,75
202212312,80,30,30
202212377,75,40,100
202212463,65,45,45
202212471,60,60,25
202212400,95,65,20
202212469,90,80,80


#### 하나의 행을 선택 

`-` 방법1

In [363]:
df.iloc[0]

att    65
rep    55
mid    50
fin    40
Name: 202212380, dtype: int64

`-` 방법2

In [364]:
df.iloc[[0]]

,att,rep,mid,fin
202212380,65,55,50,40


`-` 방법3

In [367]:
df.iloc[0,:]

att    65
rep    55
mid    50
fin    40
Name: 202212380, dtype: int64

`-` 방법4

In [368]:
df.iloc[[0],:]

,att,rep,mid,fin
202212380,65,55,50,40


`-` 방법5

In [370]:
df.loc['202212380']

att    65
rep    55
mid    50
fin    40
Name: 202212380, dtype: int64

`-` 방법6

In [373]:
df.loc[['202212380']]

,att,rep,mid,fin
202212380,65,55,50,40


`-` 방법7

In [375]:
df.loc['202212380',:]

att    65
rep    55
mid    50
fin    40
Name: 202212380, dtype: int64

`-` 방법8

In [376]:
df.loc[['202212380'],:]

,att,rep,mid,fin
202212380,65,55,50,40


`-` 방법9

In [378]:
len(df)

20

In [379]:
[True]+[False]*19

[True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False]

In [383]:
df.iloc[[True]+[False]*19]

,att,rep,mid,fin
202212380,65,55,50,40


`-` 방법10

In [384]:
df.iloc[[True]+[False]*19,:]

,att,rep,mid,fin
202212380,65,55,50,40


`-` 방법11

In [387]:
df.loc[[True]+[False]*19]

,att,rep,mid,fin
202212380,65,55,50,40


`-` 방법12

In [388]:
df.loc[[True]+[False]*19,:]

,att,rep,mid,fin
202212380,65,55,50,40


#### 여러개의 행을 선택 

`-` 방법1

In [390]:
df.iloc[[0,2],:]

,att,rep,mid,fin
202212380,65,55,50,40
202212363,65,90,60,30


In [417]:
df.iloc[[0,2]]

,att,rep,mid,fin
202212380,65,55,50,40
202212363,65,90,60,30


`-` 방법2

In [391]:
df.loc[['202212380','202212363'],:]

,att,rep,mid,fin
202212380,65,55,50,40
202212363,65,90,60,30


In [418]:
df.loc[['202212380','202212363']]

,att,rep,mid,fin
202212380,65,55,50,40
202212363,65,90,60,30


`-` 그외에 여러가지.. 

In [413]:
df.iloc[::2] # 스트라이딩

,att,rep,mid,fin
202212380,65,55,50,40
202212363,65,90,60,30
202212312,80,30,30,100
202212463,65,45,45,90
202212400,95,65,20,10
202212318,55,75,35,25
202212443,95,55,15,35
202212458,50,55,15,85
202212482,50,50,45,10
202212387,70,70,40,35


In [414]:
df.iloc[:10] # 슬라이싱

,att,rep,mid,fin
202212380,65,55,50,40
202212370,95,100,50,80
202212363,65,90,60,30
202212488,55,80,75,80
202212312,80,30,30,100
202212377,75,40,100,15
202212463,65,45,45,90
202212471,60,60,25,0
202212400,95,65,20,10
202212469,90,80,80,20


In [419]:
df.loc[:'202212469'] # 슬라이싱

,att,rep,mid,fin
202212380,65,55,50,40
202212370,95,100,50,80
202212363,65,90,60,30
202212488,55,80,75,80
202212312,80,30,30,100
202212377,75,40,100,15
202212463,65,45,45,90
202212471,60,60,25,0
202212400,95,65,20,10
202212469,90,80,80,20


In [416]:
df.iloc[[True]*10+[False]*10] # bool 

,att,rep,mid,fin
202212380,65,55,50,40
202212370,95,100,50,80
202212363,65,90,60,30
202212488,55,80,75,80
202212312,80,30,30,100
202212377,75,40,100,15
202212463,65,45,45,90
202212471,60,60,25,0
202212400,95,65,20,10
202212469,90,80,80,20


In [425]:
df.loc[[True]*10+[False]*10] # bool 

,att,rep,mid,fin
202212380,65,55,50,40
202212370,95,100,50,80
202212363,65,90,60,30
202212488,55,80,75,80
202212312,80,30,30,100
202212377,75,40,100,15
202212463,65,45,45,90
202212471,60,60,25,0
202212400,95,65,20,10
202212469,90,80,80,20


In [426]:
df.iloc[list(df.att<60)] # bool 

,att,rep,mid,fin
202212488,55,80,75,80
202212318,55,75,35,25
202212367,50,80,40,30
202212458,50,55,15,85
202212482,50,50,45,10


In [427]:
df.loc[list(df.att<60)] # bool 

,att,rep,mid,fin
202212488,55,80,75,80
202212318,55,75,35,25
202212367,50,80,40,30
202212458,50,55,15,85
202212482,50,50,45,10


`-` 아래도 가능하다.

In [430]:
df.loc[df.att<60] # bool 

,att,rep,mid,fin
202212488,55,80,75,80
202212318,55,75,35,25
202212367,50,80,40,30
202212458,50,55,15,85
202212482,50,50,45,10


`-` 그런데 아래는 에러가 난다

In [428]:
df.iloc[df.att<60] # bool 

ValueError: iLocation based boolean indexing cannot use an indexable as a mask

In [432]:
df.iloc[list(df.att<60)] # iloc입장에선 살짝 서운하긴함

,att,rep,mid,fin
202212488,55,80,75,80
202212318,55,75,35,25
202212367,50,80,40,30
202212458,50,55,15,85
202212482,50,50,45,10


### pandas 공부 3단계 ~ 

`-` 데이터시각화 시간에서!